# 07 — Grand Comparison: All Steps & Encoders

**Project:** Tomato Quality Semantic Segmentation — Background Bias Analysis
**Models:** U-Net with MobileNetV2 & EfficientNet-B0 Encoders

## What this notebook does

1. Loads all results from notebooks 04, 05, and 06
2. Produces the grand metrics table — all steps × both encoders
3. mIoU and ECE bar charts across all conditions
4. All 6 reliability diagrams in a 2×3 grid
5. Per-class IoU heatmap (6 conditions × 7 classes)
6. Training history overlay — all 3 steps on the same axes per encoder
7. Cross-background robustness chart — all 4 transfer scenarios
8. Per-class IoU grouped bar chart — all 3 steps per encoder
9. Saves all figures to OUTPUT_DIR

> This notebook contains no training. It only loads pkl files and produces figures.

## 1. Imports

In [ ]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

## 2. Load Shared Config

In [ ]:
sys.path.insert(0, str(Path('..').resolve()))
from config import *

print(f'OUTPUT_DIR : {OUTPUT_DIR}')

## 3. Load All Step Results

In [ ]:
for name, path in [
    ('step1_results.pkl',     OUTPUT_DIR / 'step1_results.pkl'),
    ('step2_results.pkl',     OUTPUT_DIR / 'step2_results.pkl'),
    ('step3_results.pkl',     OUTPUT_DIR / 'step3_results.pkl'),
    ('robustness_results.pkl',OUTPUT_DIR / 'robustness_results.pkl'),
]:
    assert path.exists(), f'Missing: {path}. Run notebooks 04-06 first.'
    print(f'  Found: {name}')

with open(OUTPUT_DIR / 'step1_results.pkl', 'rb') as f:
    s1 = pickle.load(f)
with open(OUTPUT_DIR / 'step2_results.pkl', 'rb') as f:
    s2 = pickle.load(f)
with open(OUTPUT_DIR / 'step3_results.pkl', 'rb') as f:
    s3 = pickle.load(f)
with open(OUTPUT_DIR / 'robustness_results.pkl', 'rb') as f:
    rob = pickle.load(f)

# ── Unpack into flat named variables ─────────────────────────────────────────
m_nat  = s1['mobilenet']['metrics'];      cal_m_nat  = s1['mobilenet']['calibration'];  hist_m_nat  = s1['mobilenet']['history']
m_rem  = s2['mobilenet']['metrics'];      cal_m_rem  = s2['mobilenet']['calibration'];  hist_m_rem  = s2['mobilenet']['history']
m_syn  = s3['mobilenet']['metrics'];      cal_m_syn  = s3['mobilenet']['calibration'];  hist_m_syn  = s3['mobilenet']['history']

e_nat  = s1['efficientnet']['metrics'];   cal_e_nat  = s1['efficientnet']['calibration']; hist_e_nat  = s1['efficientnet']['history']
e_rem  = s2['efficientnet']['metrics'];   cal_e_rem  = s2['efficientnet']['calibration']; hist_e_rem  = s2['efficientnet']['history']
e_syn  = s3['efficientnet']['metrics'];   cal_e_syn  = s3['efficientnet']['calibration']; hist_e_syn  = s3['efficientnet']['history']

rob_m_nat2nat   = rob['mobilenet']['nat_to_nat']
rob_m_nat2syn   = rob['mobilenet']['nat_to_synth']
rob_m_syn2nat   = rob['mobilenet']['synth_to_nat']
rob_m_syn2syn   = rob['mobilenet']['synth_to_synth']

rob_e_nat2nat   = rob['efficientnet']['nat_to_nat']
rob_e_nat2syn   = rob['efficientnet']['nat_to_synth']
rob_e_syn2nat   = rob['efficientnet']['synth_to_nat']
rob_e_syn2syn   = rob['efficientnet']['synth_to_synth']

print('All results loaded successfully.')

## 4. Grand Metrics Table

In [ ]:
rows = []
conditions = [
    ('Step 1 — Natural',   m_nat, e_nat, cal_m_nat, cal_e_nat),
    ('Step 2 — Removed',   m_rem, e_rem, cal_m_rem, cal_e_rem),
    ('Step 3 — Synthetic', m_syn, e_syn, cal_m_syn, cal_e_syn),
]

for cond, mm, me, cm, ce in conditions:
    for enc, m, cal in [('MobileNetV2', mm, cm), ('EfficientNet-B0', me, ce)]:
        rows.append({
            'Condition'  : cond,
            'Encoder'    : enc,
            'Pixel Acc'  : round(m['pixel_accuracy'], 4),
            'Mean IoU'   : round(m['mean_iou'],        4),
            'Mean Dice'  : round(m['mean_dice'],        4),
            'ECE'        : round(cal['ece'],            4),
        })

grand_df = pd.DataFrame(rows)
print('\n=== GRAND METRICS TABLE ===')
print(grand_df.to_string(index=False))
print()

# Also save as CSV for easy reference
grand_df.to_csv(OUTPUT_DIR / 'grand_metrics_table.csv', index=False)
print('Saved -> grand_metrics_table.csv')

## 5. mIoU and ECE Bar Charts — All Conditions

In [ ]:
conditions_short = [
    'Step1\nNatural', 'Step1\nNatural',
    'Step2\nRemoved', 'Step2\nRemoved',
    'Step3\nSynthetic', 'Step3\nSynthetic',
]

labels   = ['S1 Natural', 'S2 Removed', 'S3 Synthetic']
miou_mob = [m_nat['mean_iou'],  m_rem['mean_iou'],  m_syn['mean_iou']]
miou_eff = [e_nat['mean_iou'],  e_rem['mean_iou'],  e_syn['mean_iou']]
ece_mob  = [cal_m_nat['ece'],   cal_m_rem['ece'],   cal_m_syn['ece']]
ece_eff  = [cal_e_nat['ece'],   cal_e_rem['ece'],   cal_e_syn['ece']]

x = np.arange(3); width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── mIoU ─────────────────────────────────────────────────────────────────────
bars_m = axes[0].bar(x - width/2, miou_mob, width, label='MobileNetV2',
                     color='#2196F3', alpha=0.85, edgecolor='white')
bars_e = axes[0].bar(x + width/2, miou_eff, width, label='EfficientNet-B0',
                     color='#FF5722', alpha=0.85, edgecolor='white')
for bars in [bars_m, bars_e]:
    for bar in bars:
        h = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.005,
                     f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=11)
axes[0].set_ylabel('Mean IoU', fontsize=11)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('Mean IoU — All Background Conditions', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)

# ── ECE ──────────────────────────────────────────────────────────────────────
bars_m2 = axes[1].bar(x - width/2, ece_mob, width, label='MobileNetV2',
                      color='#2196F3', alpha=0.85, edgecolor='white')
bars_e2 = axes[1].bar(x + width/2, ece_eff, width, label='EfficientNet-B0',
                      color='#FF5722', alpha=0.85, edgecolor='white')
for bars in [bars_m2, bars_e2]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.001,
                     f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, fontsize=11)
axes[1].set_ylabel('ECE (lower = better)', fontsize=11)
axes[1].set_ylim(0, max(max(ece_mob), max(ece_eff)) * 1.3)
axes[1].set_title('ECE — All Background Conditions', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.suptitle('Grand Comparison — mIoU and ECE Across All Conditions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_miou_ece_barchart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_miou_ece_barchart.png')

## 6. All 6 Reliability Diagrams — 2×3 Grid

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(21, 14))

diagram_configs = [
    (axes[0, 0], cal_m_nat, 'MobileNetV2 — Step 1 Natural BG'),
    (axes[0, 1], cal_m_rem, 'MobileNetV2 — Step 2 BG Removed'),
    (axes[0, 2], cal_m_syn, 'MobileNetV2 — Step 3 Synthetic BG'),
    (axes[1, 0], cal_e_nat, 'EfficientNet-B0 — Step 1 Natural BG'),
    (axes[1, 1], cal_e_rem, 'EfficientNet-B0 — Step 2 BG Removed'),
    (axes[1, 2], cal_e_syn, 'EfficientNet-B0 — Step 3 Synthetic BG'),
]

for ax, cal, title in diagram_configs:
    m = cal['bin_counts'] > 0
    ax.bar(cal['bin_confidences'][m], cal['bin_accuracies'][m],
           width=1. / len(cal['bin_counts']), alpha=0.6,
           color='steelblue', edgecolor='navy', label='Actual accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    ax.set_xlabel('Confidence', fontsize=10)
    ax.set_ylabel('Accuracy', fontsize=10)
    ax.set_title(f'{title}\nECE = {cal["ece"]:.4f}', fontsize=11)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=9); ax.set_aspect('equal')

plt.suptitle('Reliability Diagrams — All 6 Conditions (2 Encoders × 3 Steps)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_all_reliability_diagrams.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_all_reliability_diagrams.png')

## 7. Per-Class IoU Heatmap — 6 Conditions × 7 Classes

In [ ]:
row_labels = [
    'MobileNetV2 — Natural',
    'MobileNetV2 — Removed',
    'MobileNetV2 — Synthetic',
    'EfficientNet-B0 — Natural',
    'EfficientNet-B0 — Removed',
    'EfficientNet-B0 — Synthetic',
]

iou_matrix = np.array([
    np.nan_to_num(m_nat['iou_per_class'], nan=0.),
    np.nan_to_num(m_rem['iou_per_class'], nan=0.),
    np.nan_to_num(m_syn['iou_per_class'], nan=0.),
    np.nan_to_num(e_nat['iou_per_class'], nan=0.),
    np.nan_to_num(e_rem['iou_per_class'], nan=0.),
    np.nan_to_num(e_syn['iou_per_class'], nan=0.),
])

fig, ax = plt.subplots(figsize=(15, 7))
im = ax.imshow(iou_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label='IoU')

ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=11)
ax.set_yticks(range(6))
ax.set_yticklabels(row_labels, fontsize=11)

for i in range(6):
    for j in range(NUM_CLASSES):
        v = iou_matrix[i, j]
        text_color = 'black' if 0.3 < v < 0.75 else 'white'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=10, color=text_color, fontweight='bold')

# Separator between encoders
ax.axhline(2.5, color='white', linewidth=3)

ax.set_title('Per-Class IoU Heatmap — All Steps × Both Encoders',
             fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_per_class_iou_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_per_class_iou_heatmap.png')

## 8. Training History Overlay — All 3 Steps per Encoder

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

STEP_COLORS = {'Natural': '#2196F3', 'Removed': '#FF5722', 'Synthetic': '#4CAF50'}
STEP_STYLES = {'Natural': '-', 'Removed': '--', 'Synthetic': '-.'}

for enc_idx, (enc_label, histories) in enumerate([
    ('MobileNetV2',     [('Natural',   hist_m_nat),
                         ('Removed',   hist_m_rem),
                         ('Synthetic', hist_m_syn)]),
    ('EfficientNet-B0', [('Natural',   hist_e_nat),
                         ('Removed',   hist_e_rem),
                         ('Synthetic', hist_e_syn)]),
]):
    ax_loss = axes[enc_idx, 0]
    ax_miou = axes[enc_idx, 1]

    for step_name, hist in histories:
        epochs = range(1, len(hist['val_loss']) + 1)
        col = STEP_COLORS[step_name]
        sty = STEP_STYLES[step_name]
        ax_loss.plot(epochs, hist['train_loss'], color=col, linestyle=sty,
                     alpha=0.5, linewidth=1.2)
        ax_loss.plot(epochs, hist['val_loss'],   color=col, linestyle=sty,
                     linewidth=2, label=f'{step_name} Val')
        ax_miou.plot(epochs, hist['train_miou'], color=col, linestyle=sty,
                     alpha=0.5, linewidth=1.2)
        ax_miou.plot(epochs, hist['val_miou'],   color=col, linestyle=sty,
                     linewidth=2, label=f'{step_name} Val')

    ax_loss.set_xlabel('Epoch', fontsize=10)
    ax_loss.set_ylabel('Loss', fontsize=10)
    ax_loss.set_title(f'{enc_label} — Loss (faded=train, solid=val)', fontsize=11, fontweight='bold')
    ax_loss.legend(fontsize=9)

    ax_miou.set_xlabel('Epoch', fontsize=10)
    ax_miou.set_ylabel('Mean IoU', fontsize=10)
    ax_miou.set_title(f'{enc_label} — Mean IoU (faded=train, solid=val)', fontsize=11, fontweight='bold')
    ax_miou.legend(fontsize=9)

plt.suptitle('Training History Overlay — All Steps × Both Encoders',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_training_history_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_training_history_overlay.png')

## 9. Cross-Background Robustness Chart

In [ ]:
scenario_labels = [
    'Baseline\nNat→Nat',
    '(a) Nat→Syn',
    '(b) Syn→Nat',
    '(c) Syn→Syn',
]

miou_m_rob = [
    rob_m_nat2nat['mean_iou'],
    rob_m_nat2syn['mean_iou'],
    rob_m_syn2nat['mean_iou'],
    rob_m_syn2syn['mean_iou'],
]
miou_e_rob = [
    rob_e_nat2nat['mean_iou'],
    rob_e_nat2syn['mean_iou'],
    rob_e_syn2nat['mean_iou'],
    rob_e_syn2syn['mean_iou'],
]

x = np.arange(4); width = 0.35
fig, ax = plt.subplots(figsize=(13, 6))

bars_m = ax.bar(x - width/2, miou_m_rob, width, label='MobileNetV2',
                color='#2196F3', alpha=0.85, edgecolor='white')
bars_e = ax.bar(x + width/2, miou_e_rob, width, label='EfficientNet-B0',
                color='#FF5722', alpha=0.85, edgecolor='white')

for bars in [bars_m, bars_e]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
                f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(scenario_labels, fontsize=11)
ax.set_ylabel('Mean IoU', fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title('Cross-Background Robustness — mIoU Across Transfer Scenarios',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Annotate drop from baseline to (a) and (b)
for enc_idx, (miou_list, color) in enumerate([
    (miou_m_rob, '#2196F3'),
    (miou_e_rob, '#FF5722'),
]):
    drop_a = miou_list[0] - miou_list[1]
    drop_b = miou_list[0] - miou_list[2]
    offset = -0.18 + enc_idx * 0.35
    ax.annotate(f'drop: {drop_a:.3f}',
                xy=(1 + offset, miou_list[1]),
                xytext=(1 + offset, miou_list[1] - 0.08),
                ha='center', fontsize=8, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_robustness_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_robustness_chart.png')

## 10. Per-Class IoU — All 3 Steps per Encoder

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 7))
x     = np.arange(NUM_CLASSES)
width = 0.25

STEP_COLORS3 = ['#2196F3', '#FF5722', '#4CAF50']
STEP_LABELS3 = ['Step 1: Natural', 'Step 2: Removed', 'Step 3: Synthetic']

for ax, (m1, m2, m3), enc_label in [
    (axes[0], (m_nat, m_rem, m_syn), 'MobileNetV2'),
    (axes[1], (e_nat, e_rem, e_syn), 'EfficientNet-B0'),
]:
    for i, (m, col, lbl) in enumerate(zip([m1, m2, m3], STEP_COLORS3, STEP_LABELS3)):
        iou_vals = np.nan_to_num(m['iou_per_class'], nan=0.)
        bars = ax.bar(x + (i - 1) * width, iou_vals, width,
                      label=lbl, color=col, alpha=0.85, edgecolor='white')
        for bar, v in zip(bars, iou_vals):
            if v > 0.03:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f'{v:.2f}', ha='center', va='bottom', fontsize=7, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('IoU', fontsize=10)
    ax.set_ylim(0, 1.25)
    ax.set_title(f'{enc_label} — Per-Class IoU: All 3 Steps',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Per-Class IoU — Steps 1 vs 2 vs 3 × Both Encoders',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_per_class_iou_grouped.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_per_class_iou_grouped.png')

## 11. Pixel Accuracy and Dice Score Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

labels   = ['S1 Natural', 'S2 Removed', 'S3 Synthetic']
pa_mob   = [m_nat['pixel_accuracy'], m_rem['pixel_accuracy'], m_syn['pixel_accuracy']]
pa_eff   = [e_nat['pixel_accuracy'], e_rem['pixel_accuracy'], e_syn['pixel_accuracy']]
dice_mob = [m_nat['mean_dice'],      m_rem['mean_dice'],      m_syn['mean_dice']]
dice_eff = [e_nat['mean_dice'],      e_rem['mean_dice'],      e_syn['mean_dice']]

x = np.arange(3); width = 0.35

for ax, vals_m, vals_e, ylabel, title in [
    (axes[0], pa_mob,   pa_eff,   'Pixel Accuracy', 'Pixel Accuracy — All Conditions'),
    (axes[1], dice_mob, dice_eff, 'Mean Dice',      'Mean Dice — All Conditions'),
]:
    bm = ax.bar(x - width/2, vals_m, width, label='MobileNetV2',
                color='#2196F3', alpha=0.85, edgecolor='white')
    be = ax.bar(x + width/2, vals_e, width, label='EfficientNet-B0',
                color='#FF5722', alpha=0.85, edgecolor='white')
    for bars in [bm, be]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                    f'{h:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)

plt.suptitle('Pixel Accuracy and Dice Score — Grand Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_pixel_acc_dice.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> grand_pixel_acc_dice.png')

## 12. Cross-Background Robustness — Full Table

In [ ]:
rob_rows = []
for enc, r_nn, r_ns, r_sn, r_ss in [
    ('MobileNetV2',
     rob_m_nat2nat, rob_m_nat2syn, rob_m_syn2nat, rob_m_syn2syn),
    ('EfficientNet-B0',
     rob_e_nat2nat, rob_e_nat2syn, rob_e_syn2nat, rob_e_syn2syn),
]:
    for scenario, r in [
        ('Baseline: Natural → Natural', r_nn),
        ('(a) Natural → Synthetic',     r_ns),
        ('(b) Synthetic → Natural',     r_sn),
        ('(c) Synthetic → Synthetic',   r_ss),
    ]:
        rob_rows.append({
            'Encoder'   : enc,
            'Scenario'  : scenario,
            'mIoU'      : round(r['mean_iou'],        4),
            'Pixel Acc' : round(r['pixel_accuracy'],  4),
            'Mean Dice' : round(r['mean_dice'],        4),
        })

rob_df = pd.DataFrame(rob_rows)
print('\n=== CROSS-BACKGROUND ROBUSTNESS TABLE ===')
print(rob_df.to_string(index=False))
print()
rob_df.to_csv(OUTPUT_DIR / 'robustness_table.csv', index=False)
print('Saved -> robustness_table.csv')

## 13. Summary & File Manifest

In [ ]:
print('=' * 70)
print('  NOTEBOOK 07 — GRAND COMPARISON COMPLETE')
print('=' * 70)
print()
print(f'  {"Condition":<28} {"Encoder":<18} {"mIoU":>8} {"ECE":>8} {"Dice":>8}')
print(f'  {"-"*72}')
for cond, mm, me, cm, ce in [
    ('Step 1 — Natural',   m_nat, e_nat, cal_m_nat, cal_e_nat),
    ('Step 2 — Removed',   m_rem, e_rem, cal_m_rem, cal_e_rem),
    ('Step 3 — Synthetic', m_syn, e_syn, cal_m_syn, cal_e_syn),
]:
    print(f'  {cond:<28} {"MobileNetV2":<18} '
          f'{mm["mean_iou"]:>8.4f} {cm["ece"]:>8.4f} {mm["mean_dice"]:>8.4f}')
    print(f'  {"" :<28} {"EfficientNet-B0":<18} '
          f'{me["mean_iou"]:>8.4f} {ce["ece"]:>8.4f} {me["mean_dice"]:>8.4f}')
    print()

print()
print('  Figures saved:')
figure_files = [
    'grand_miou_ece_barchart.png',
    'grand_all_reliability_diagrams.png',
    'grand_per_class_iou_heatmap.png',
    'grand_training_history_overlay.png',
    'grand_robustness_chart.png',
    'grand_per_class_iou_grouped.png',
    'grand_pixel_acc_dice.png',
]
for fname in figure_files:
    fp = OUTPUT_DIR / fname
    status = 'OK' if fp.exists() else 'MISSING'
    print(f'    [{status}] {fname}')

print()
print('  CSV tables saved:')
for fname in ['grand_metrics_table.csv', 'robustness_table.csv']:
    fp = OUTPUT_DIR / fname
    status = 'OK' if fp.exists() else 'MISSING'
    print(f'    [{status}] {fname}')

print()
print('  Project complete. All results ready for thesis.')
print('=' * 70)

In [1]:
!jupyter nbconvert --to html 07_grand_comparison.ipynb 

[NbConvertApp] Converting notebook 07_grand_comparison.ipynb to html
[NbConvertApp] Writing 392892 bytes to 07_grand_comparison.html
